# LLMs & LCEL: Working with Language Models in LangChain

This notebook covers:
- Initializing LLMs with `init_chat_model` (provider-agnostic)
- Provider-specific setup (OpenAI, Anthropic)
- Working with message objects (`HumanMessage`, `SystemMessage`, `AIMessage`)
- **LCEL** (LangChain Expression Language): the `|` pipe operator
- Chain execution modes: `invoke`, `batch`, `stream`
- Schema inspection

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# Verify keys are loaded
print("OpenAI key set:", bool(os.getenv("OPENAI_API_KEY")))
print("Anthropic key set:", bool(os.getenv("ANTHROPIC_API_KEY")))

## 1. Initializing LLMs

### `init_chat_model` — the provider-agnostic way

`init_chat_model` is the **recommended universal initializer**. You swap providers by changing the `model` string — the rest of your code stays the same.

Key parameters:
- `model`: model name (e.g. `"gpt-4o-mini"`, `"claude-sonnet-4-5-20250929"`)
- `model_provider`: explicit provider (`"openai"`, `"anthropic"`) — auto-detected if omitted
- `temperature`: creativity (0=deterministic, 1=creative)
- `streaming`: enable streaming mode
- `max_retries`: auto-retry on rate-limit errors

In [ ]:
from langchain.chat_models import init_chat_model

# Universal initialization — works for any supported provider
llm = init_chat_model(
    model="gpt-4o-mini",
    temperature=0.7,
    streaming=True,
    max_retries=3,
)

response = llm.invoke("What is the capital of France? Answer in one word.")
print(f"Response: {response.content}")
print(f"Type: {type(response)}")

### Comparing multiple models

Because `init_chat_model` normalizes initialization, you can loop over model names and compare outputs side-by-side.

In [ ]:
prompt = "Explain recursion in one sentence."

models = {
    "gpt-4o-mini": init_chat_model(model="gpt-4o-mini", temperature=0.7),
    "gpt-4o":      init_chat_model(model="gpt-4o", temperature=0.7),
}

if os.getenv("ANTHROPIC_API_KEY"):
    models["claude-sonnet"] = init_chat_model(
        model="claude-sonnet-4-5-20250929",
        model_provider="anthropic",
        temperature=0.7,
    )

print(f"Prompt: {prompt}\n")
for name, model in models.items():
    resp = model.invoke(prompt)
    print(f"[{name}]: {resp.content}\n")

### Provider-specific initialization

For production, you may want explicit provider classes for type safety, IDE auto-complete, and provider-specific parameters (e.g. `timeout`, `max_tokens`).

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic

# OpenAI — explicit class with extra params
openai_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7,
    max_tokens=1500,
    timeout=30,        # hard timeout per request
    max_retries=3,
)

# Anthropic
if os.getenv("ANTHROPIC_API_KEY"):
    anthropic_model = ChatAnthropic(model="claude-sonnet-4-5-20250929")
    print("Both providers initialized")
else:
    print("Only OpenAI initialized (no Anthropic key)")

## 2. Message Types

LangChain uses structured message objects rather than plain strings:

| Class | Role | Typical Use |
|---|---|---|
| `SystemMessage` | `system` | Personality / behavior instructions |
| `HumanMessage`  | `user`   | User input |
| `AIMessage`     | `assistant` | Previous model responses |

Passing a list of messages is the basis of **multi-turn conversations**.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Turn 1
messages = [
    SystemMessage(content="You are a pirate. Always answer like a pirate."),
    HumanMessage(content="What's the weather like today?"),
]
response = model.invoke(messages)
print(f"Turn 1: {response.content}")

# Turn 2 — append response and ask a follow-up
messages.append(response)   # append AIMessage to conversation
messages.append(HumanMessage(content="What about tomorrow?"))

response2 = model.invoke(messages)
print(f"\nTurn 2: {response2.content}")

## 3. LCEL — The Pipe Operator (`|`)

**LangChain Expression Language (LCEL)** composes Runnables using `|`:

```
chain = prompt | model | parser
```

Each component is a `Runnable` with `.invoke()`, `.batch()`, and `.stream()`. The output of one Runnable becomes the input to the next.

Key packages:
- `langchain_core.prompts.ChatPromptTemplate` — structures the prompt
- `langchain_openai.ChatOpenAI` — the LLM
- `langchain_core.output_parsers.StrOutputParser` — parses AIMessage → plain string

In [ ]:
import json
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# --- Define each component ---
prompt = ChatPromptTemplate.from_template(
    "You are a helpful assistant. Answer in one sentence: {question}"
)
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
parser = StrOutputParser()

# --- Compose with the pipe operator ---
chain = prompt | model | parser

# --- invoke: single input ---
result = chain.invoke({"question": "What is LangChain?"})
print(f"invoke result type: {type(result).__name__}")
print(f"Response: {result}")

## 4. Execution Modes: `invoke`, `batch`, `stream`

Every LCEL chain supports three execution modes:

| Method | Use case |
|---|---|
| `.invoke(input)` | Single request, synchronous |
| `.batch([input1, input2, ...])` | Multiple requests in parallel |
| `.stream(input)` | Token-by-token streaming |


In [ ]:
# --- batch: run multiple inputs ---
translate_prompt = ChatPromptTemplate.from_template("Translate to French: {text}")
translate_chain = translate_prompt | model | parser

inputs = [
    {"text": "Hello, how are you?"},
    {"text": "What is your name?"},
    {"text": "Where is the nearest restaurant?"},
]
results = translate_chain.batch(inputs)

print("Batch results:")
for inp, out in zip(inputs, results):
    print(f"  {inp['text']} => {out}")

In [ ]:
# --- stream: token-by-token output ---
haiku_prompt = ChatPromptTemplate.from_template("Write a haiku about: {topic}")
haiku_chain = haiku_prompt | model | parser

print("Streaming output: ", end="")
for chunk in haiku_chain.stream({"topic": "nature"}):
    print(chunk, end="", flush=True)
print()  # newline after

## 5. Schema Inspection

Every LCEL chain exposes `input_schema` and `output_schema` — useful for documentation, validation, and debugging.

> The schema is derived from the first and last Runnable in the chain.

In [ ]:
summary_prompt = ChatPromptTemplate.from_template("Summarize: {text}")
summary_chain = summary_prompt | model | parser

print("Input Schema:")
print(json.dumps(summary_chain.input_schema.model_json_schema(), indent=2))

print("\nOutput Schema:")
print(json.dumps(summary_chain.output_schema.model_json_schema(), indent=2))

## 6. Exercise: Multi-Model Response Aggregator

Build a function that:
1. Accepts a `question` and list of `model_names`
2. Queries all models using `init_chat_model`
3. Returns a `dict[model_name → response]`

In [ ]:
def get_responses(question: str, model_names: list[str]) -> dict[str, str]:
    responses = {}
    for name in model_names:
        m = init_chat_model(model=name, temperature=0.7)
        responses[name] = m.invoke(question).content
    return responses

results = get_responses("What is AI?", ["gpt-4o-mini", "gpt-4o"])
for model_name, answer in results.items():
    print(f"[{model_name}]: {answer[:120]}...\n")